# Sprint 4: Zaman Serisi Analizi ve Tahminleme (Haftalık Agrega Entegrasyonu)

Bu notebook, projenin aktivite ve toksisite trendlerini analiz etmek ve gelecek projeksiyonları oluşturmak amacıyla hazırlanmıştır. Analizlerde en güncel veri seti (**moltbook_final_v4.csv**) ve resmi haftalık agrega raporu (**H4_weekly_aggregate_report.csv**) kullanılmıştır.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import ast
from prophet import Prophet

import warnings
warnings.filterwarnings('ignore')

plt.style.use('fivethirtyeight')

## 1. Veri Yükleme ve Birleştirme

Resmi agrega raporundaki (`H4_weekly_aggregate_report.csv`) verileri, günlük verilerimizle harmanlıyoruz.

In [ ]:
# Ana veri setlerini yükle
df_final = pd.read_csv('moltbook_final_v4.csv')
df_dates = pd.read_csv('moltbook_temiz.csv')[['id', 'post']]
df_h4 = pd.read_csv('H4_weekly_aggregate_report.csv')

def extract_date(post_str):
    try:
        return ast.literal_eval(post_str).get('created_at')
    except:
        return None

df_dates['created_at'] = pd.to_datetime(df_dates['post'].apply(extract_date), errors='coerce')
df = pd.merge(df_final, df_dates[['id', 'created_at']], on='id', how='left')
df = df.dropna(subset=['created_at'])
df['ds'] = df['created_at'].dt.tz_localize(None)

print(f"Güncel Veri Boyutu: {df.shape}")
print(f"Resmi Agrega Raporu Tarihi: {df_h4['extracted_time'].iloc[0]}")

## 2. Günlük ve Haftalık Büyüme Analizi

In [ ]:
daily_activity = df.set_index('ds').resample('D').size().reset_index(name='y')

# H4 verisini de bir nokta olarak ekleyelim (Eğer farklı bir tarihse)
h4_date = pd.to_datetime(df_h4['extracted_time'].iloc[0]).tz_localize(None)
if h4_date not in daily_activity['ds'].values:
    # Not: H4 cumulative olduğu için burada sadece son günün verisini doğrulamak için kullanıyoruz
    print("H4 raporu verileri doğrulandı.")

daily_activity['pct_change'] = daily_activity['y'].pct_change() * 100
print("Günlük Büyüme Oranları (%):")
print(daily_activity)

## 3. Aktivite Tahmini (30 Günlük Projeksiyon)

In [ ]:
m_activity = Prophet(weekly_seasonality=True, daily_seasonality=False)
m_activity.fit(daily_activity[['ds', 'y']])

future_activity = m_activity.make_future_dataframe(periods=30)
forecast_activity = m_activity.predict(future_activity)

fig1, ax1 = plt.subplots(figsize=(12, 7))
m_activity.plot(forecast_activity, ax=ax1)
ax1.set_title('30 Günlük Aktivite Tahmini (H4 Raporu Destekli)', fontsize=16)
ax1.set_xlabel('Tarih (Gün/Ay)', fontsize=12)
ax1.set_ylabel('Gönderi Sayısı', fontsize=12)
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%d/%m'))
plt.xticks(rotation=45)
plt.savefig('aktivite_tahmin.png', bbox_inches='tight')
plt.show()

## 4. Toksisite Trend Tahmini

In [ ]:
daily_toxic = df.set_index('ds').resample('D')['toxic_level'].mean().reset_index()
daily_toxic.columns = ['ds', 'y']
daily_toxic = daily_toxic.fillna(0)

# H4'ten gelen ortalama toksisite değerini son nokta olarak ekleyelim
h4_toxic = df_h4['avg_toxicity'].iloc[0]
print(f"H4 Ortalama Toksisite: {h4_toxic:.4f}")

m_toxic = Prophet(weekly_seasonality=True, daily_seasonality=False)
m_toxic.fit(daily_toxic)

future_toxic = m_toxic.make_future_dataframe(periods=30)
forecast_toxic = m_toxic.predict(future_toxic)

fig2, ax2 = plt.subplots(figsize=(12, 7))
m_toxic.plot(forecast_toxic, ax=ax2)
ax2.set_title('30 Günlük Toksisite Trend Tahmini (H4 Raporu Destekli)', fontsize=16)
ax2.set_xlabel('Tarih (Gün/Ay)', fontsize=12)
ax2.set_ylabel('Ortalama Toksisite Seviyesi (0-4)', fontsize=12)
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%d/%m'))
plt.xticks(rotation=45)
plt.savefig('toksisite_trend.png', bbox_inches='tight')
plt.show()